# Shark Knowledge Medallion — Pipeline End-to-End

Demostración completa del pipeline de ingeniería de datos de la arquitectura medallón (Bronze → Silver → Merge → Warehouse → Gold), incluyendo ejemplos de lotes, idempotencia y especies fuera del lote semilla.

---

## 1. Flujo end-to-end del pipeline

La función `run_pipeline()` en `src/pipeline.py` orquesta las cinco etapas secuencialmente:

1. **Bronze** — Fetch de GBIF Taxonomy, GBIF Occurrence y Wikipedia; persistencia como JSON crudo con envelope de metadatos.
2. **Silver** — Validación contra contratos Pydantic v2; registros válidos a `valid/`, rechazados a `rejected/` con motivo explícito.
3. **Merge** — Wikipedia como pivote 1:1 + GBIF Taxonomy (solo EXACT) + Occurrence agregada (N→1).
4. **Warehouse** — UPSERT idempotente en DuckDB (5 tablas: `silver_wikipedia`, `silver_gbif_taxonomy`, `silver_gbif_occurrences`, `species_merged`, `species_catalog`).
5. **Gold** — Construcción de Species (12 secciones canónicas), chunking semántico, embeddings BAAI/bge-small-en-v1.5, índice FAISS.

El lote semilla de 18 especies se divide en **Lote 1** (primeras 9) y **Lote 2** (últimas 9) solo como convención de demostración.

In [ ]:
import sys
sys.path.append("..")

from src.pipeline import run_pipeline
from src.config.seed_species import SEED_SPECIES
from src.resolver.slugify import slugify
from src.warehouse.loader import Warehouse
import json

In [ ]:
print("Lote semilla completo:")
for i, s in enumerate(SEED_SPECIES, 1):
    print(f"  {i:2d}. {s['species_key']:35s} | {s['scientific_name']:30s}")

lote_1 = [s["scientific_name"] for s in SEED_SPECIES[:9]]
lote_2 = [s["scientific_name"] for s in SEED_SPECIES[9:]]
print(f"\nLote 1 ({len(lote_1)} especies):")
for n in lote_1:
    print(f"  - {n}")
print(f"\nLote 2 ({len(lote_2)} especies):")
for n in lote_2:
    print(f"  - {n}")

---
## 2. Ejemplo de uso de pipeline usando Lote 1

Ejecutamos el pipeline con las primeras 9 especies del lote semilla. El pipeline devuelve un resumen con el estado de cada etapa.

In [ ]:
print(">>> Ejecutando pipeline con Lote 1 (primeras 9 especies seed)\n")
result_1 = run_pipeline(lote_1, batch_id="demo_lote_1")

print("=== RESUMEN DEL PIPELINE - LOTE 1 ===")
print(json.dumps(result_1, indent=2, default=str))

### Análisis del resumen

- **Bronze**: Se crearon archivos JSON crudos para cada fuente (gbif_taxonomy, gbif_occurrence, wikipedia).
- **Silver**: Wikipedia y GBIF Taxonomy deberían ser válidos para todas (match EXACT). Ocurrencias pueden tener algunos rechazos por coordenadas faltantes.
- **Merge**: Todas las especies con Wikipedia válida se fusionan.
- **Warehouse**: Las 5 tablas se pueblan con UPSERT. `species_catalog` registra las 9 especies con `origin = 'SEED'`.
- **Gold**: Se generan chunks, embeddings e índice FAISS.

In [ ]:
# Consultar el catálogo de especies después de Lote 1
wh = Warehouse()
wh.connect()
conn = wh.connect()
catalog_1 = conn.execute(
    "SELECT species_key, scientific_name, origin, first_processed_at, last_processed_at "
    "FROM species_catalog ORDER BY species_key"
).fetchall()
print(">>> species_catalog después de Lote 1:")
for row in catalog_1:
    print(f"  {row[0]:35s} | origin={row[2]:6s} | first={row[3]} | last={row[4]}")

---
## 3. Ejemplo de uso de pipeline usando Lote 2

Ejecutamos el pipeline con las últimas 9 especies del lote semilla.

In [ ]:
print(">>> Ejecutando pipeline con Lote 2 (últimas 9 especies seed)\n")
result_2 = run_pipeline(lote_2, batch_id="demo_lote_2")

print("=== RESUMEN DEL PIPELINE - LOTE 2 ===")
print(json.dumps(result_2, indent=2, default=str))

In [ ]:
# Verificar que ahora tenemos 18 especies en el catálogo
catalog_2 = conn.execute(
    "SELECT species_key, scientific_name, origin FROM species_catalog ORDER BY species_key"
).fetchall()
print(f">>> species_catalog después de Lote 2: {len(catalog_2)} especies\n")
for row in catalog_2:
    print(f"  {row[0]:35s} | origin={row[2]:6s}")

---
## 4. Ejemplo de idempotencia usando Lote 1

**Idempotencia**: reprocesar exactamente el mismo lote debe producir `filas_nuevas = 0` en todas las tablas del warehouse. Las filas existentes se actualizan (UPSERT) pero no se duplican.

In [ ]:
print(">>> Reprocesando Lote 1 (segunda corrida) - prueba de idempotencia\n")
result_1_idem = run_pipeline(lote_1, batch_id="demo_lote_1_idempotent")

print("=== EVIDENCIA DE IDEMPOTENCIA - LOTE 1 ===")
print(f"{'Tabla':30s} | {'Antes':>6s} | {'Después':>7s} | {'Nuevas':>6s} | {'Actualizadas':>12s}")
print("-" * 70)
for ev in result_1_idem["warehouse"]:
    if "filas_antes" in ev:
        print(f"{ev['tabla']:30s} | {ev['filas_antes']:6d} | {ev['filas_despues']:7d} | {ev['filas_nuevas']:6d} | {ev['filas_actualizadas']:12d}")
    else:
        print(f"{ev['tabla']:30s} | {ev['count']:6d}")

print("\n✅ Idempotencia verificada: filas_nuevas = 0 en todas las tablas (los datos ya existían)")

---
## 5. Ejemplo de idempotencia usando Lote 2

Misma prueba de idempotencia pero con el segundo lote.

In [ ]:
print(">>> Reprocesando Lote 2 (segunda corrida) - prueba de idempotencia\n")
result_2_idem = run_pipeline(lote_2, batch_id="demo_lote_2_idempotent")

print("=== EVIDENCIA DE IDEMPOTENCIA - LOTE 2 ===")
print(f"{'Tabla':30s} | {'Antes':>6s} | {'Después':>7s} | {'Nuevas':>6s} | {'Actualizadas':>12s}")
print("-" * 70)
for ev in result_2_idem["warehouse"]:
    if "filas_antes" in ev:
        print(f"{ev['tabla']:30s} | {ev['filas_antes']:6d} | {ev['filas_despues']:7d} | {ev['filas_nuevas']:6d} | {ev['filas_actualizadas']:12d}")
    else:
        print(f"{ev['tabla']:30s} | {ev['count']:6d}")

print("\n✅ Idempotencia verificada: filas_nuevas = 0 en todas las tablas")

---
## 6. Ejemplos de uso de pipeline usando species para cinco especies fuera de la lista SEED

El universo de especies es **abierto**: podemos procesar cualquier nombre científico, no solo los del lote semilla. El sistema resuelve automáticamente el título de Wikipedia vía búsqueda y redirects.

Especies nuevas (no SEED) se registran en `species_catalog` con `origin = 'AD_HOC'`.

In [ ]:
# 5 especies fuera del lote semilla
ad_hoc_species = [
    "Manta birostris",       # Giant oceanic manta ray
    "Pristis pristis",       # Largetooth sawfish
    "Mobula mobular",        # Devil fish
    "Dasyatis americana",    # Southern stingray
    "Rhinoptera bonasus",    # Cownose ray
]

print(">>> Especies AD-HOC a procesar:")
for s in ad_hoc_species:
    sk = slugify(s)
    print(f"  {s:25s} -> {sk}")

In [ ]:
print(">>> Ejecutando pipeline con 5 especies AD-HOC\n")
result_adhoc = run_pipeline(ad_hoc_species, batch_id="demo_adhoc")

print("=== RESUMEN DEL PIPELINE - ESPECIES AD-HOC ===")
print(json.dumps(result_adhoc, indent=2, default=str))

In [ ]:
# Verificar que las nuevas especies se registraron con origin = 'AD_HOC'
catalog_adhoc = conn.execute(
    "SELECT species_key, scientific_name, origin, first_seen_batch_id "
    "FROM species_catalog WHERE origin = 'AD_HOC' ORDER BY species_key"
).fetchall()
print(f">>> Especies AD-HOC en species_catalog: {len(catalog_adhoc)}\n")
for row in catalog_adhoc:
    print(f"  {row[0]:25s} | {row[1]:25s} | origin={row[2]:6s} | batch={row[3]}")

### Verificación de catálogo completo

Ahora el catálogo contiene las 18 especies seed + las 5 especies ad-hoc = 23 especies.

In [ ]:
total = conn.execute("SELECT COUNT(*) FROM species_catalog").fetchone()[0]
print(f"Total de especies en catálogo: {total}")
print(f"  SEED:  {conn.execute("SELECT COUNT(*) FROM species_catalog WHERE origin = 'SEED'").fetchone()[0]}")
print(f"  AD_HOC: {conn.execute("SELECT COUNT(*) FROM species_catalog WHERE origin = 'AD_HOC'").fetchone()[0]}")

---
## 7. Ejemplo de idempotencia usando las cinco especies de la sección 6

Reprocesamos las mismas 5 especies AD-HOC para verificar idempotencia también con especies fuera del lote semilla.

In [ ]:
print(">>> Reprocesando 5 especies AD-HOC (segunda corrida) - prueba de idempotencia\n")
result_adhoc_idem = run_pipeline(ad_hoc_species, batch_id="demo_adhoc_idempotent")

print("=== EVIDENCIA DE IDEMPOTENCIA - ESPECIES AD-HOC ===")
print(f"{'Tabla':30s} | {'Antes':>6s} | {'Después':>7s} | {'Nuevas':>6s} | {'Actualizadas':>12s}")
print("-" * 70)
for ev in result_adhoc_idem["warehouse"]:
    if "filas_antes" in ev:
        print(f"{ev['tabla']:30s} | {ev['filas_antes']:6d} | {ev['filas_despues']:7d} | {ev['filas_nuevas']:6d} | {ev['filas_actualizadas']:12d}")
    else:
        print(f"{ev['tabla']:30s} | {ev['count']:6d}")

print("\n✅ Idempotencia verificada: filas_nuevas = 0 (especies AD-HOC tampoco se duplican)")

---
## Resumen de resultados

| Sección | Descripción | Resultado esperado |
|---------|-------------|-------------------|
| 2 | Pipeline con Lote 1 (9 seed) | 9 especies procesadas, catálogo poblado con origin=SEED |
| 3 | Pipeline con Lote 2 (9 seed) | 9 especies adicionales, 18 total en catálogo |
| 4 | Idempotencia Lote 1 | filas_nuevas = 0 en todas las tablas |
| 5 | Idempotencia Lote 2 | filas_nuevas = 0 en todas las tablas |
| 6 | Pipeline con 5 especies AD-HOC | Nuevas especies registradas con origin=AD_HOC |
| 7 | Idempotencia AD-HOC | filas_nuevas = 0, confirmando idempotencia universal |

**Conclusión**: El pipeline es determinista, idempotente y soporta un universo abierto de especies. El catálogo en DuckDB (`species_catalog`) es el registro maestro creciente de todas las especies procesadas.

---
*Shark Knowledge Medallion — Pipeline End-to-End*